# <a id='toc1_'></a>[Datenqualität (intern) 📉](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [Datenqualität (intern) 📉](#toc1_)    
  - [Datenstand 🕥](#toc1_1_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

<br>

## <a id='toc1_1_'></a>[Datenstand 🕥](#toc0_)

In [1]:
import os
from pathlib import Path
import pandas as pd
import datetime as dt
from pandas_plots import tbl, pls, hlp
import duckdb as ddb

from connection_helper import sql
import numpy as np

hlp.show_package_version(["plotly"])

dir_db=Path("C://temp") if hlp.get_os(hlp.OperatingSystem.WINDOWS) else Path(os.path.expanduser("~/tmp"))

file_epi = dir_db/'2025-06-20_data_epi.duckdb'
file_meta = dir_db/'2025-06-20_meta_epi.duckdb'

if not file_epi.exists() or not file_meta.exists():
    raise Exception(f"files {file_epi} or {file_meta} not found")

URL_V2 = 'https://gitlab.opencode.de/robert-koch-institut/zentrum-fuer-krebsregisterdaten/cancerdata-references/-/raw/main/data/v2/'

os.environ['THEME']='light'

🐍 3.12.8 | 📦 plotly: 6.3.1 | 📦 pandas: 2.3.3 | 📦 numpy: 1.26.4 | 📦 duckdb: 1.4.1 | 📦 pandas-plots: 0.19.1 | 📦 connection-helper: 0.13.1


In [2]:
batch_min=399

In [3]:
sql.print_meta(file_epi)
# print(f"\n{'latest batch:': <25}{batch_max}")

sqlite db file:          2025-06-20_data_epi.duckdb
data tag:                epi2024_2
sql table created:       2025-06-20 15:31:26
document created:        2025-10-09 11:22:09


In [4]:
con = ddb.connect()
con.execute("PRAGMA disable_progress_bar;")
con.execute(f"ATTACH DATABASE '{file_epi}' AS epi (READ_ONLY);")
con.execute(f"ATTACH DATABASE '{file_meta}' AS meta (READ_ONLY); set schema 'epi';")

In [5]:
Batches = con.sql("select * from meta.Batches;")
Checks = con.sql("select * from meta.Checks;")
ColumnCheck = con.sql("select * from meta.ColumnCheck;")
Frequencies = con.sql("select * from meta.Frequencies;")
# _meta_meta = con.sql("select * from meta._meta;")

df_checks = Checks.to_df().astype({"EKRNR": str})
df_freqs = (
    Frequencies.project(
        "*, concat(cast(BatchID as varchar),' | ', cast(cast(BatchZeit as date) as varchar)) as batch_label"
    )
    .to_df()
    .astype({"EKRNR": str})
)
# df_columnchecks= ColumnCheck.to_df().set_index('ColumnCheckID')
# df_checks = df_checks.join(df_columnchecks, on='ColumnCheckID')
# df_checks["check_label"] = df_checks.apply(lambda x: f"{x['Alias']}[{x['ColumnCheckID']}]", axis=1)
# df_checks= df_checks[["EKRNR","Anzahl","BatchID","DJahr","check_label","ColumnCheckID"]]

In [6]:
db_epi = con.sql("select * from Tumor4;").project("*, left(ICDGM10,3) as icd10_3d")

In [ ]:
batch_max=df_checks['BatchID'].max()

(ColumnCheck
    .filter("isTracked and not isInactive")
    .project("ColumnCheckID, Alias, Description")
    .order("Alias")
    .to_df()
    )

,ColumnCheckID,Alias,Description
0,83,A_EKRNR_GKZ_unplausibel,<html>Laut GKZ kommt der Patient nicht aus dem...
1,66,A_ICD10_SEX_fehlerhaft,<html>Männer mit frauentypischen Tumoren (ICD1...
2,54,A_ICD10_keineAuswertung,<html>Die Diagnose ist nicht gemäß ICD-10-WHO ...
3,37,A_Mehrfachmeldung,<html>Die Meldung wurde als Mehrfachmeldung zu...
4,1,A_SEX_fehlerhaft,"<html>Die Variable ""Geschlecht"" ist leer oder ..."
5,58,A_Zeitangaben_fehlerhaft,<html>Entweder Geburts- und/oder Diagnosejahr ...
6,67,B_DALT_HISC_ICD10_unplausibel,<html>Das Diagnosealter passt nicht zur Histol...
7,77,B_DALT_unplausibel,"<html>Unwahrscheinlich hohes Patientenalter, w..."
8,39,B_DCO_DDIMP_inkonsistent,<html>Fälle zum gleichen Patienten mit gleiche...
9,70,B_DIG_HISC_unplausibel,<html>Die Kombination aus Dignität und Histolo...


In [9]:
db_checks_all_bl = Checks.filter(f"BatchID >= 390 and ColumnCheckID <> 48").aggregate("ColumnCheckID::text as id, BatchID::text as batch, sum(Anzahl) as Anzahl")

_=pls.plot_stacked_bars(db_checks_all_bl.to_df(), swap = True, renderer="")

In [13]:
tbl.describe_df(Checks.project("* exclude (ID)").to_df(),'checks', fig_cols=5)

🔵 *** df: checks ***  
🟣 shape: (331_297, 5)
🟣 duplicates: 85_873  
🟠 column stats all (dtype | uniques | missings) [values]  
- index [0, 1, 2, 3, 4,]  
- EKRNR (int32 | 11 | 0 (0%)) [1, 2, 3, 4, 5,]  
- ColumnCheckID (int32 | 34 | 0 (0%)) [1, 12, 14, 20, 37,]  
- Anzahl (int32 | 4_347 | 0 (0%)) [1, 2, 3, 4, 5,]  
- BatchID (int32 | 141 | 0 (0%)) [73, 95, 119, 128, 132,]  
- DJahr (object | 13 | 3_588 (1%)) ['2013', '2014', '2015', '2016', '2017',]  
🟠 column stats numeric  

column        |    count    |  min   |  lower  |   q25   | median  |   mean    |   q75   |  upper  |      max      |    std     |   cv   |        sum        |  skew  |   kurto  
--------------+-------------+--------+---------+---------+---------+-----------+---------+---------+---------------+------------+--------+-------------------+--------+----------
EKRNR         | 331_297.000 |  1.000 |   1.000 |   3.000 |   5.000 |     5.757 |   9.000 |  11.000 |        11.000 |      3.113 |  0.541 |     1_907_412.000 |  0.

,EKRNR,ColumnCheckID,Anzahl,BatchID,DJahr
0,6,52,7,412,alt
1,5,52,1,412,alt
2,6,52,54,412,alt
